# Metabolomics overview plot w/ Sirius Canopus classifications

Date created: 12/28/2024

Notebook author: Yang Chen

Data analysis by: Britta De Pessemier

This notebook plots the following:
- Metabolomics overview plots showing number of total metabolites, with and without suspects library, and classified annotations w/ Sirius Canopus classifications

In [194]:
# Import Python packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib_venn import venn3
import plotly.graph_objects as go
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib import colormaps
from upsetplot import from_indicators, plot
from upsetplot import from_memberships, UpSet

In [195]:
# Read in Sirius Canopus classifications file
sirius_result = pd.read_csv('../Data/metabolomics/Run3_10252024/canopus_structure_summary.csv')
sirius_result

,formulaRank,molecularFormula,adduct,precursorFormula,NPC#pathway,NPC#pathway Probability,NPC#superclass,NPC#superclass Probability,NPC#class,NPC#class Probability,...,ClassyFire#most specific class,ClassyFire#most specific class Probability,ClassyFire#all classifications,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,featureId,overallFeatureQuality
0,1,C6H15N5O3,[M + K]+,C6H15KN5O3+,Carbohydrates,0.897,Small peptides,0.649,Aminoacids,0.364,...,Alpha amino acids and derivatives,0.532,Organic compounds; Organoheterocyclic compound...,244.079,30,0.501,638448045036744826,637772591529933629,168,NaN
1,1,C4H6N3O2,[M + Na]+,C4H6N3NaO2+,Alkaloids,0.852,Pseudoalkaloids (transamidation),0.286,Purine alkaloids,0.372,...,Azoles,0.534,Organic compounds; Organoheterocyclic compound...,151.035,30,0.501,638448053119168779,637772593396399054,264,NaN
2,1,C6H15NO3,[M + H]+,C6H16NO3+,Carbohydrates,0.504,Aminosugars and aminoglycosides,0.372,Aminosugars,0.334,...,"1,2-aminoalcohols",0.896,Organic compounds; Alcohols and polyols; Organ...,150.112,31,0.509,638448044789280881,637772593182489535,261,NaN
3,19,C8H16N6O8,[M - H2O + H]+,C8H15N6O7+,Carbohydrates,0.996,Aminosugars and aminoglycosides,0.537,Aminosugars,0.198,...,Pentoses,0.533,Organic compounds; Organoheterocyclic compound...,307.102,31,0.511,638448046974513380,637772589617330870,37,NaN
4,1,C5H11N3O2,[M + H]+,C5H12N3O2+,Amino acids and Peptides,0.774,Small peptides,0.805,Aminoacids,0.910,...,Gamma amino acids and derivatives,0.844,Organic compounds; Lipids and lipid-like molec...,146.092,31,0.511,638448093829083581,637772594541444126,313,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5620,2,C30H44O,[M + K]+,C30H44KO+,Shikimates and Phenylpropanoids,0.462,Flavonoids,0.266,Chalcones,0.230,...,Phenylpropanes,0.887,Organic compounds; Ketones; Organooxygen compo...,459.301,506,8.431,638475783681175681,637772836619926454,32963,NaN
5621,1,C30H38N2O2,[M + H3N + H]+,C30H42N3O2+,Alkaloids,0.252,Anthranilic acid alkaloids,0.096,Acridone alkaloids,0.162,...,Phenylpropanes,0.965,Organic compounds; Organic acids and derivativ...,476.328,506,8.434,638475786025792128,637772836590566321,32962,NaN
5622,2,C31H49NO,[M + K]+,C31H49KNO+,Alkaloids,0.526,Pseudoalkaloids (transamidation),0.416,Phenylalanine-derived alkaloids,0.185,...,Phenylpropanes,0.783,Organic compounds; Organonitrogen compounds; O...,490.344,506,8.434,638475808943472943,637772836645092283,32964,NaN
5623,1,C28H57N3O7,[M + H]+,C28H58N3O7+,Amino acids and Peptides,0.660,Small peptides,0.182,Aminoacids,0.123,...,Amino acids and derivatives,0.563,"Organic compounds; Amino acids, peptides, and ...",548.425,507,8.453,638475808628900005,637772836926110696,33038,NaN


In [196]:
# Take a look at all the column names
sirius_result.columns

Index(['formulaRank', 'molecularFormula', 'adduct', 'precursorFormula',
       'NPC#pathway', 'NPC#pathway Probability', 'NPC#superclass',
       'NPC#superclass Probability', 'NPC#class', 'NPC#class Probability',
       'ClassyFire#superclass', 'ClassyFire#superclass probability',
       'ClassyFire#class', 'ClassyFire#class Probability',
       'ClassyFire#subclass', 'ClassyFire#subclass Probability',
       'ClassyFire#level 5', 'ClassyFire#level 5 Probability',
       'ClassyFire#most specific class',
       'ClassyFire#most specific class Probability',
       'ClassyFire#all classifications', 'ionMass', 'retentionTimeInSeconds',
       'retentionTimeInMinutes', 'formulaId', 'alignedFeatureId', 'featureId',
       'overallFeatureQuality'],
      dtype='object')

In [197]:
# Check that the number of features is both unique and equal to the number of rows in the DataFrame
if sirius_result['featureId'].nunique() == sirius_result.shape[0]:
    print(f"There are {sirius_result['featureId'].nunique()} different metabolites.")
else:
    print("The number of unique features does not match the number of rows in the Sirius result.")

There are 5625 different metabolites.


In [198]:
### Check that the n=5625 metabolites from Sirius are all within the total n=7683 metabolites

# Read the Sirius features into a set
sirius_features = set(sirius_result['featureId'])

# Read the info_feature_complete dataset
info_feature_complete = pd.read_csv('../Data/metabolomics/Run3_10252024/info_feature_complete.csv')

# Total number of metabolites in info_feature_complete
total_num_metabolites = info_feature_complete.shape[0]

# Check if all Sirius features are in the complete set
sirius_in_total = sirius_features.issubset(info_feature_complete['Feature'])

# Display results
if sirius_in_total:
    print(f"All {len(sirius_features)} metabolites from Sirius are within the total of {total_num_metabolites} metabolites.")
else:
    # Find how many Sirius features are missing
    missing_features = sirius_features - set(info_feature_complete['featureId'])
    print(f"{len(missing_features)} metabolites from Sirius are not in the total of {total_num_metabolites} metabolites.")


All 5625 metabolites from Sirius are within the total of 7683 metabolites.


In [199]:
### Check how many of the n=1027 metabolites from GNPS suspect spectral library are within the Sirius result of n=5625

# Expecting most of them to be here...though it's possible Sirius was able to give decently confident predictions that GNPS did not at all as they rely on separate spectral databases

# Read in GNPS2 metabolites table obtained WITH suspects library
gnps_with_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps.tsv', sep='\t')

# Find the intersection between the two columns
matching_scans = gnps_with_suspects['#Scan#'].isin(sirius_result['featureId'])

# Count the number of matches
count = matching_scans.sum()

# Calculate the percentage of matches
percentage = (count / gnps_with_suspects.shape[0]) * 100

# Display the result
print(f"Number of #Scan# values in gnps_with_suspects that are within sirius_result['featureId']: {count}, {percentage:.2f}%")

Number of #Scan# values in gnps_with_suspects that are within sirius_result['featureId']: 918, 89.39%


In [200]:
# Set probability threshold
# (reccomendation of 0.75 from the Dorrestein lab)
prob_thres = 0.75

In [201]:
### Check probability calls at various levels for NPC designations

# Count the total number of rows
total_rows = sirius_result.shape[0]

# Define the columns to check
columns = ['NPC#pathway Probability', 'NPC#superclass Probability', 'NPC#class Probability']

# Iterate through the columns, calculate counts and percentages, and display results
for col in columns:
    count = sirius_result[sirius_result[col] >= prob_thres].shape[0]
    percentage = (count / total_rows) * 100
    print(f"Number of rows with {col} >= {prob_thres}: {count}, {percentage:.2f}%")


Number of rows with NPC#pathway Probability >= 0.75: 3222, 57.28%
Number of rows with NPC#superclass Probability >= 0.75: 1982, 35.24%
Number of rows with NPC#class Probability >= 0.75: 1363, 24.23%


In [202]:
### Check probability calls at various levels for ClassyFire designations

# Count the total number of rows
total_rows = sirius_result.shape[0]

# Define the columns to check for ClassyFire designations
columns = [
    'ClassyFire#superclass probability', 
    'ClassyFire#class Probability', 
    'ClassyFire#subclass Probability', 
    'ClassyFire#level 5 Probability', 
    'ClassyFire#most specific class Probability'
]

# Iterate through the columns, calculate counts and percentages, and display results
for col in columns:
    count = sirius_result[sirius_result[col] >= prob_thres].shape[0]
    percentage = (count / total_rows) * 100
    print(f"Number of rows with {col} >= {prob_thres}: {count}, {percentage:.2f}%")


Number of rows with ClassyFire#superclass probability >= 0.75: 4519, 80.34%
Number of rows with ClassyFire#class Probability >= 0.75: 3857, 68.57%
Number of rows with ClassyFire#subclass Probability >= 0.75: 2233, 39.70%
Number of rows with ClassyFire#level 5 Probability >= 0.75: 1379, 24.52%
Number of rows with ClassyFire#most specific class Probability >= 0.75: 1855, 32.98%


In [203]:
### Check how many features have probability of >= 0.75 across ALL the classyfire designations/columns

# Filter rows where all specified columns have values >= prob_thres
filtered_df = sirius_result[(sirius_result[columns] >= prob_thres).all(axis=1)]

# Count the number of rows that meet the condition
count = filtered_df.shape[0]

# Calculate the percentage of such rows
percentage = (count / sirius_result.shape[0]) * 100

# Display the result
print(f"Number of features with probability >= {prob_thres} across all ClassyFire designations: {count}, {percentage:.2f}%")

Number of features with probability >= 0.75 across all ClassyFire designations: 968, 17.21%


In [204]:
### Check how many features have probability of >= 0.75 across the 3 main classification levels: superclass, class, subclass

# Filter rows where the first three ClassyFire columns have values >= prob_thres
filtered_df = sirius_result[
    (sirius_result['ClassyFire#superclass probability'] >= prob_thres) &
    (sirius_result['ClassyFire#class Probability'] >= prob_thres) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
]

# Count and percentage
count = filtered_df.shape[0]
percentage = (count / sirius_result.shape[0]) * 100

# Display the result
print(f"Number of features with probability >= {prob_thres} across the first three ClassyFire designations: {count}, {percentage:.2f}%")

Number of features with probability >= 0.75 across the first three ClassyFire designations: 2136, 37.97%


In [205]:
# Count non-NaN values for each column
non_nan_counts = sirius_result[['ClassyFire#superclass', 'ClassyFire#class', 'ClassyFire#subclass']].notna().sum()

# Display results
for column, count in non_nan_counts.items():
    percentage = (count / sirius_result.shape[0]) * 100
    print(f"Number of non-NaN values in {column}: {count}, {percentage:.2f}%")

Number of non-NaN values in ClassyFire#superclass: 5625, 100.00%
Number of non-NaN values in ClassyFire#class: 5347, 95.06%
Number of non-NaN values in ClassyFire#subclass: 4126, 73.35%


In [206]:
# Filter rows where columns are not NaN and probabilities are >= prob_thres
filtered_counts = {
    'ClassyFire#superclass': sirius_result[
        sirius_result['ClassyFire#superclass'].notna() &
        (sirius_result['ClassyFire#superclass probability'] >= prob_thres)
    ].shape[0],
    'ClassyFire#class': sirius_result[
        sirius_result['ClassyFire#class'].notna() &
        (sirius_result['ClassyFire#class Probability'] >= prob_thres)
    ].shape[0],
    'ClassyFire#subclass': sirius_result[
        sirius_result['ClassyFire#subclass'].notna() &
        (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
    ].shape[0]
}

# Display results
for column, count in filtered_counts.items():
    percentage = (count / sirius_result.shape[0]) * 100
    print(f"Number of non-NaN values in {column} with probability >= {prob_thres}: {count}, {percentage:.2f}%")

Number of non-NaN values in ClassyFire#superclass with probability >= 0.75: 4519, 80.34%
Number of non-NaN values in ClassyFire#class with probability >= 0.75: 3857, 68.57%
Number of non-NaN values in ClassyFire#subclass with probability >= 0.75: 2233, 39.70%


In [207]:
# Filter rows where all classification levels are not NaN and all probabilities are >= prob_thres (must be true for ALL)
filtered_df = sirius_result[
    sirius_result['ClassyFire#superclass'].notna() &
    sirius_result['ClassyFire#class'].notna() &
    sirius_result['ClassyFire#subclass'].notna() &
    (sirius_result['ClassyFire#superclass probability'] >= prob_thres) &
    (sirius_result['ClassyFire#class Probability'] >= prob_thres) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
]

# Count the number of rows that meet the condition
count = filtered_df.shape[0]

# Calculate the percentage of such rows
percentage = (count / sirius_result.shape[0]) * 100

# Display the result
print(f"Number of rows with all classification levels not NaN and probabilities >= {prob_thres}: {count}, {percentage:.2f}%")

Number of rows with all classification levels not NaN and probabilities >= 0.75: 2136, 37.97%


In [208]:
### Check how many of the n=1027 metabolites from GNPS suspect spectral library are within the Sirius result of n=2136

# Expecting *majority?* of them to be here...though it's possible Sirius was able to give decently confident predictions that GNPS did not at all as they rely on separate spectral databases
# Probably less than 89% though from the all Sirius comparison above

# Read in GNPS2 metabolites table obtained WITH suspects library
gnps_with_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps.tsv', sep='\t')

# Find the intersection between the two columns
matching_scans = gnps_with_suspects['#Scan#'].isin(filtered_df['featureId'])

# Count the number of matches
count = matching_scans.sum()

# Calculate the percentage of matches
percentage = (count / gnps_with_suspects.shape[0]) * 100

# Display the result
print(f"Number of #Scan# values in gnps_with_suspects that are within Sirius all main 3 levels not NaN and prob >= 0.75: {count}, {percentage:.2f}%")

Number of #Scan# values in gnps_with_suspects that are within Sirius all main 3 levels not NaN and prob >= 0.75: 473, 46.06%


### GNPS results

In [209]:
# Read in table of all spectra and output number of total metabolites
info_feature_complete = pd.read_csv('../Data/metabolomics/Run3_10252024/info_feature_complete.csv')
total_num_metabolites = info_feature_complete.shape[0]
print(f'Total number of consensus MS2 spectra metabolites from GNPS2: ' + str(total_num_metabolites))

Total number of consensus MS2 spectra metabolites from GNPS2: 7683


In [210]:
# Read in GNPS2 metabolites table obtained WITHOUT suspects library
gnps_without_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps_withoutsuspects.tsv', sep='\t')
num_gnps_without_suspects = gnps_without_suspects.shape[0]
print(f'Number of GNPS2 outputed metabolites WITHOUT suspects library: ' + str(num_gnps_without_suspects))

Number of GNPS2 outputed metabolites WITHOUT suspects library: 432


In [211]:
filtered_df = sirius_result[
    sirius_result['ClassyFire#superclass'].notna() &
    sirius_result['ClassyFire#class'].notna() &
    sirius_result['ClassyFire#subclass'].notna() &
    (sirius_result['ClassyFire#superclass probability'] >= prob_thres) &
    (sirius_result['ClassyFire#class Probability'] >= prob_thres) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
]

# Find the intersection between the two columns
matching_scans = gnps_without_suspects['#Scan#'].isin(filtered_df['featureId'])

# Count the number of matches
count = matching_scans.sum()

# Calculate the percentage of matches
tot_annotations_sirius = 5625
percentage = (count / tot_annotations_sirius) * 100

# Display the result
print(f"Number of #Scan# values in num_gnps_without_suspects that are within all Sirius predicted']: {count}, {percentage:.2f}%")

Number of #Scan# values in num_gnps_without_suspects that are within all Sirius predicted']: 235, 4.18%


In [212]:
# Read in GNPS2 metabolites table obtained WITH suspects library
gnps_with_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps.tsv', sep='\t')
num_gnps_with_suspects = gnps_with_suspects.shape[0]
print(f'Number of GNPS2 outputed metabolites WITH suspects library: ' + str(num_gnps_with_suspects))

Number of GNPS2 outputed metabolites WITH suspects library: 1027


In [213]:
# Count rows where 'superclass' is not NaN
num_annotated_metabolites = gnps_with_suspects['superclass'].notna().sum()

print(f"Number of superclass annotated metabolites: {num_annotated_metabolites}")

Number of superclass annotated metabolites: 250


### Nested Venn Diagram showing number of metabolites from GNPS and Sirius

In [215]:
# Filter rows where all classification levels are not NaN and probabilities are >= prob_thres (must be true for ALL)
filtered_df = sirius_result[
    sirius_result['ClassyFire#superclass'].notna() &
    sirius_result['ClassyFire#class'].notna() &
    sirius_result['ClassyFire#subclass'].notna() &
    (sirius_result['ClassyFire#superclass probability'] >= prob_thres) &
    (sirius_result['ClassyFire#class Probability'] >= prob_thres) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
]

# Define updated circle sizes
size1 = total_num_metabolites  # Size of Circle 1 (Total MS2 spectra identified)
size2 = sirius_result.shape[0]  # Size of Circle 2 (Sirius metabolites)
size3 = filtered_df.shape[0]  # Size of Circle 3 (Sirius with confident classifications)
size4 = num_gnps_with_suspects  # Size of Circle 4 (GNPS suspect spectral library matches)
size5 = num_gnps_without_suspects  # Size of Circle 5 (GNPS non-suspect spectral library matches)

# Calculate radii based on the square root of the sizes (scaled for visualization)
scaling_factor = 0.005  # Adjust this for appropriate visualization

radius1 = np.sqrt(size1) * scaling_factor  # Radius for Circle 1
radius2 = np.sqrt(size2) * scaling_factor  # Radius for Circle 2
radius3 = np.sqrt(size3) * scaling_factor  # Radius for Circle 3
radius4 = np.sqrt(size4) * scaling_factor  # Radius for Circle 4 (GNPS suspect)
radius5 = np.sqrt(size5) * scaling_factor  # Radius for Circle 5

# Calculate the percentage for each circle relative to the total size
percent2 = (size2 / size1) * 100
percent3 = (size3 / size1) * 100
percent4 = (size4 / size1) * 100
# percent5 = (size5 / size1) * 100

# Create a figure
fig, ax = plt.subplots(figsize=(6,6))

# Create circles with correct placements to represent overlap
circle1 = plt.Circle((0.5, 0.5), radius1, facecolor='#000000', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size1} total MS2 spectra identified')
circle2 = plt.Circle((0.5, 0.5), radius2, facecolor='#00305d', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size2} total annotations from Sirius4')
circle3 = plt.Circle((0.42, 0.4), radius3, facecolor='#0096FF', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size3} annotations from Sirius4 (prob >= 0.75)')
# Note that of the 1027 from GNPS Suspect Library, 89% were within the 5625 total from Sirius, and 46% were within the 2136 of Sirius prob >= 0.75

# GNPS suspect spectral library circle (slightly shifted to show partial overlap)
circle4 = plt.Circle((0.54, 0.24), radius4, facecolor='#ADD8E6', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size4} annotations from GNPS2 Suspect Spectral Library')
# circle5 = plt.Circle((0.5, 0.1), radius5, facecolor='#000080', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size5} annotations from GNPS2 non-suspect spectral library')

# Add the circles to the plot
ax.add_patch(circle1)
ax.add_patch(circle2)
ax.add_patch(circle3)
ax.add_patch(circle4)
# ax.add_patch(circle5)

# Add percentage text inside the circles
ax.text(0.505, 0.68, f'{percent2:.1f}%', ha='center', va='center', fontsize=10, color='white')
ax.text(0.4105, 0.45, f'{percent3:.1f}%', ha='center', va='center', fontsize=10, color='white')
ax.text(0.54, 0.26, f'{percent4:.1f}%', ha='center', va='center', fontsize=10, color='black')
# ax.text(0.505, 0.13, f'{percent5:.1f}%', ha='center', va='center', fontsize=10, color='white')

# Set limits to ensure all circles are fully visible
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Set aspect ratio to be equal to ensure circles are circular
ax.set_aspect('equal')

# Remove all borders
for spine in ax.spines.values():
    spine.set_visible(False)

# Remove ticks and labels
ax.set_xticks([])
ax.set_yticks([])

# Add legend for the circles with their respective sizes and place it outside the plot
ax.legend(loc='upper left', bbox_to_anchor=(0.05, 0), fontsize=10)

# Add title with better font style and size
plt.title("Metabolites Identified from Workflow", loc='center', fontsize=16)

# Adjust layout to make room for the legend
plt.tight_layout()

# Save figure
plt.savefig('../Figures/metabolomics_Figures/overview/ms2_venn_diagram_with_sirius.png', dpi=600)